REPOSITORY CODE

In [23]:
# -*- coding: utf-8 -*-
"""
Created on Thu Dec  8 16:12:24 2022

@author: ie20391
"""

class edgar_data:
    '''
    To extract data from EDGAR SEC to python dictionary. Attributes of class are:
        - submissions_url - to show link to submissions;
        - data_url - to show link to EDGAR facts;
        - about_company - dict with information about a company;
        - available_data - pd.DataFrame with company's reports;
        - facts - the facts dictionary from EDGAR SEC;
    '''
    
    def __init__(self, company, hdr):
        #cik needs to be of len 10
        self.hdr = hdr
        if type(company) == str:
            self.ticker = company
            self.cik = edgar_data.find_cik(self)
        else:
            self.cik = company
        if len(str(self.cik)) < 10:
                self.cik = str('CIK' + ('0'*(10-len(str(self.cik))))+str(self.cik))
        else:
            raise ValueError('Non-parsable CIK')
        if self.hdr['user-agent'] == None:
            raise ValueError('''Please, specify user-agent requirements, as described in:
                                 
                                 https://www.sec.gov/os/accessing-edgar-data
                                 
                             ''')
        
        self.submissions_url = f"https://data.sec.gov/submissions/{self.cik}.json"
        self.data_url = f'https://data.sec.gov/api/xbrl/companyfacts/{self.cik}.json'
        self.about_company = edgar_data.company_info(self)
        self.available_data = edgar_data.library(self)
        self.facts = edgar_data.access_facts(self)
        
    
    def find_cik(self):
        import requests
        
        ciks_all = requests.get('https://www.sec.gov/files/company_tickers.json',
                                headers = self.hdr).json()
        ciks_all = dict([(val['ticker'], val['cik_str']) for key, val in ciks_all.items()])
        return ciks_all[self.ticker]
    
    def library(self):
        import json
        import requests
        import pandas as pd
        data = json.loads(requests.get(self.submissions_url, headers = self.hdr)._content.decode('utf8'))
        #if there are old filings available, then return df containing all finlings
        if len(data['filings']['files']) !=0:
            prior_submission_number = data['filings']['files'][0]['name']
            old_data = json.loads(requests.get(f'https://data.sec.gov/submissions/{prior_submission_number}',headers = self.hdr
                                               )._content.decode('utf8'))
            df = pd.DataFrame(data = {'rep_date':data['filings']['recent']['reportDate'] + old_data['reportDate'],
                                      'filling_date':data['filings']['recent']['filingDate'] + old_data['filingDate'],
                                      'form':data['filings']['recent']['form'] + old_data['form'],
                                      'xbrl':data['filings']['recent']['isXBRL'] + old_data['isXBRL'],
                                      'accession_num':data['filings']['recent']['accessionNumber'] + old_data['accessionNumber']})
            df = df[((df.form == '10-K') | (df.form == '10-Q')) & (df.xbrl == 1)].sort_values(by = ['rep_date'])
            df.index = [i for i in range(len(df))]
            return df
        else:
            df = pd.DataFrame(data = {'rep_date':data['filings']['recent']['reportDate'],
                                'filling_date':data['filings']['recent']['filingDate'],
                              'form':data['filings']['recent']['form'],
                              'xbrl':data['filings']['recent']['isXBRL'],
                              'accession_num':data['filings']['recent']['accessionNumber']})
        
            df = df[((df.form == '10-K') | (df.form == '10-Q')) & (df.xbrl == 1)].sort_values(by = ['rep_date'])
            df.index = [i for i in range(len(df))]
            return df
    
    def company_info(self):
        import json
        import requests
        data = json.loads(requests.get(self.submissions_url, headers = self.hdr)._content.decode('utf8'))
        d = {}
        if len(data['tickers']) > 0:    
            d['ticker'] = data['tickers'][0]
        else:
            d['ticker'] = None
        if len(data['name']) > 0:
            d['name'] = data['name']
        else:
            d['name'] = None
        if len(data['sic']) > 0:
            d['sic'] = data['sic']
        else:
            d['sic'] = None
        if len(data['sicDescription']) > 0:
            d['sic_descr'] = data['sicDescription']
        else:
            d['sic_descr'] = None
        return d
        
        
    def access_facts(self):
        import json
        import requests
        
        data = json.loads(requests.get(self.data_url, headers = self.hdr)._content.decode('utf8'))
        keys = list(data['facts'].keys())
        
        if 'us-gaap' in keys and 'ifrs-full' in keys:
            if len(data['facts']['us-gaap']) > len(data['facts']['ifrs-full']):
                data = data['facts']['us-gaap']
            else:
                data = data['facts']['ifrs-full']
            return data
        elif 'us-gaap' in keys and 'ifrs-full' not in keys:
            return data['facts']['us-gaap']
        elif 'us-gaap' not in keys and 'ifrs-full' in keys:
            return data['facts']['ifrs-full']
        
if __name__ == '__main__':
    edgar_data

DEFINE "hdr" AND "apple" to get an instance

In [24]:
from pprint import pprint

# Paste your edgar_data class definition here

# SEC requires a proper user-agent, for example:
hdr = {
    "user-agent": "Jon jonmcgowan15@gmail.com"
}

# Create an instance for Apple
apple = edgar_data("AAPL", hdr)

# Show basic company info
print("Company Info:")
pprint(apple.about_company)

# Show available filings
print("\nAvailable filings:")
print(apple.available_data.head())

# Show available financial facts keys
print("\nFacts keys:")
print(list(apple.facts.keys())[:20])  # first 20 keys

Company Info:
{'name': 'Apple Inc.',
 'sic': '3571',
 'sic_descr': 'Electronic Computers',
 'ticker': 'AAPL'}

Available filings:
     rep_date filling_date  form  xbrl         accession_num
0  2009-06-27   2009-07-22  10-Q     1  0001193125-09-153165
1  2009-09-26   2009-10-27  10-K     1  0001193125-09-214859
2  2009-12-26   2010-01-25  10-Q     1  0001193125-10-012085
3  2010-03-27   2010-04-21  10-Q     1  0001193125-10-088957
4  2010-06-26   2010-07-21  10-Q     1  0001193125-10-162840

Facts keys:
['AccountsPayable', 'AccountsPayableCurrent', 'AccountsReceivableNetCurrent', 'AccruedIncomeTaxesCurrent', 'AccruedIncomeTaxesNoncurrent', 'AccruedLiabilities', 'AccruedLiabilitiesCurrent', 'AccruedMarketingCostsCurrent', 'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment', 'AccumulatedOtherComprehensiveIncomeLossAvailableForSaleSecuritiesAdjustmentNetOfTax', 'AccumulatedOtherComprehensiveIncomeLossCumulativeChangesInNetGainLossFromCashFlowHedgesEffectNetOfTax', '

GET APPLE'S NET INCOME

In [25]:
#Check the different ways revenue might be represented

net_income_tags = [tag for tag in apple.facts.keys() if 'netincome' in tag.lower()]
print(net_income_tags)

['NetIncomeLoss', 'OtherComprehensiveIncomeLossReclassificationAdjustmentForSaleOfSecuritiesIncludedInNetIncomeNetOfTax', 'OtherComprehensiveIncomeLossReclassificationAdjustmentOnDerivativesIncludedInNetIncomeNetOfTax', 'OtherComprehensiveIncomeLossReclassificationAdjustmentOnDerivativesIncludedInNetIncomeTax', 'OtherComprehensiveIncomeReclassificationAdjustmentOnDerivativesIncludedInNetIncomeNetOfTax', 'OtherComprehensiveIncomeReclassificationAdjustmentOnDerivativesIncludedInNetIncomeTax']


In [26]:
import pandas as pd

def fact_df(fact_name):
    if fact_name in apple.facts:
        df = pd.DataFrame(apple.facts[fact_name]['units']['USD'])
        df['fact'] = fact_name
        return df
    return pd.DataFrame()

# Tags de Net Income
tags = ['NetIncomeLoss']

df_netincome = pd.concat([fact_df(tag) for tag in tags], ignore_index=True)

# Calcular tipo de periodo basado en duración
df_netincome['start'] = pd.to_datetime(df_netincome['start'])
df_netincome['end'] = pd.to_datetime(df_netincome['end'])
df_netincome['days'] = (df_netincome['end'] - df_netincome['start']).dt.days
df_netincome['Period Type'] = df_netincome['days'].apply(lambda x: 'Annual' if x > 300 else 'Quarterly')

# Limpiar duplicados: mantener el mayor Net Income por periodo
df_netincome = df_netincome.sort_values('val', ascending=False) \
                           .drop_duplicates(subset=['start', 'end', 'Period Type'])

# Ordenar final y renombrar columnas
df_netincome = df_netincome.sort_values('end')
df_netincome = df_netincome[['start', 'end', 'val', 'fact', 'Period Type', 'fp', 'form']]
df_netincome.columns = ['Period Start', 'Period End', 'Net Income (USD)', 'Tag Used', 'Period Type', 'FP', 'Form']

# Añadir columna con la duración del periodo en días
df_netincome['Period Length (days)'] = (df_netincome['Period End'] - df_netincome['Period Start']).dt.days

df_netincome[:10]



,Period Start,Period End,Net Income (USD),Tag Used,Period Type,FP,Form,Period Length (days)
0,2006-10-01,2007-09-29,3496000000,NetIncomeLoss,Annual,FY,10-K,363
3,2008-03-30,2008-06-28,1072000000,NetIncomeLoss,Quarterly,Q3,10-Q,90
2,2007-09-30,2008-06-28,3698000000,NetIncomeLoss,Quarterly,Q3,10-Q,272
5,2007-09-30,2008-09-27,6119000000,NetIncomeLoss,Annual,FY,10-K/A,363
8,2008-09-28,2008-12-27,2255000000,NetIncomeLoss,Quarterly,FY,10-K,90
9,2008-09-28,2009-03-28,3875000000,NetIncomeLoss,Quarterly,Q2,10-Q,181
10,2008-12-28,2009-03-28,1620000000,NetIncomeLoss,Quarterly,Q2,10-Q,90
13,2008-09-28,2009-06-27,5703000000,NetIncomeLoss,Quarterly,Q3,10-Q,272
15,2009-03-29,2009-06-27,1828000000,NetIncomeLoss,Quarterly,Q3,10-Q,90
18,2008-09-28,2009-09-26,8235000000,NetIncomeLoss,Annual,FY,10-K/A,363


In [27]:
#Seperate revenue between annual and quarterly

df_annual = df_netincome[df_netincome['Period Type'] == 'Annual'].reset_index(drop=True)
df_quarterly = df_netincome[df_netincome['Period Type'] == 'Quarterly'].reset_index(drop=True)

In [28]:
df_annual[:20]

,Period Start,Period End,Net Income (USD),Tag Used,Period Type,FP,Form,Period Length (days)
0,2006-10-01,2007-09-29,3496000000,NetIncomeLoss,Annual,FY,10-K,363
1,2007-09-30,2008-09-27,6119000000,NetIncomeLoss,Annual,FY,10-K/A,363
2,2008-09-28,2009-09-26,8235000000,NetIncomeLoss,Annual,FY,10-K/A,363
3,2009-09-27,2010-09-25,14013000000,NetIncomeLoss,Annual,FY,10-K,363
4,2010-09-26,2011-09-24,25922000000,NetIncomeLoss,Annual,FY,10-K,363
5,2011-09-25,2012-09-29,41733000000,NetIncomeLoss,Annual,None,8-K,370
6,2012-09-30,2013-09-28,37037000000,NetIncomeLoss,Annual,FY,10-K,363
7,2013-09-29,2014-09-27,39510000000,NetIncomeLoss,Annual,FY,10-K,363
8,2014-09-28,2015-09-26,53394000000,NetIncomeLoss,Annual,FY,10-K,363
9,2015-09-27,2016-09-24,45687000000,NetIncomeLoss,Annual,FY,10-K,363


In [29]:
#df_quartery contain quarter data with a period of days superior of 90 days, which should be the case for quarterly reports, therefore, we remove instances with period length is over 90 days

df_quarterly = df_quarterly[df_quarterly['Period Length (days)'] <= 95]
df_quarterly[:20]

,Period Start,Period End,Net Income (USD),Tag Used,Period Type,FP,Form,Period Length (days)
0,2008-03-30,2008-06-28,1072000000,NetIncomeLoss,Quarterly,Q3,10-Q,90
2,2008-09-28,2008-12-27,2255000000,NetIncomeLoss,Quarterly,FY,10-K,90
4,2008-12-28,2009-03-28,1620000000,NetIncomeLoss,Quarterly,Q2,10-Q,90
6,2009-03-29,2009-06-27,1828000000,NetIncomeLoss,Quarterly,Q3,10-Q,90
7,2009-06-28,2009-09-26,2532000000,NetIncomeLoss,Quarterly,FY,10-K,90
8,2009-09-27,2009-12-26,3378000000,NetIncomeLoss,Quarterly,Q1,10-Q,90
9,2009-12-27,2010-03-27,3074000000,NetIncomeLoss,Quarterly,Q2,10-Q,90
11,2010-03-28,2010-06-26,3253000000,NetIncomeLoss,Quarterly,FY,10-K,90
13,2010-06-27,2010-09-25,4308000000,NetIncomeLoss,Quarterly,FY,10-K,90
14,2010-09-26,2010-12-25,6004000000,NetIncomeLoss,Quarterly,FY,8-K,90


ADD THE REVENUE OF THE 4 QUARTERS OF EACH FISCAL YEAR TO GET YEARLY REVENUE, ALSO GOOD TO COMPARED TO df_annual

In [30]:
# 1. Make sure Period End is datetime
df_quarterly['Period End'] = pd.to_datetime(df_quarterly['Period End'])

# 2. Determine Apple's fiscal year
df_quarterly['Fiscal Year'] = df_quarterly['Period End'].apply(
    lambda d: d.year if d.month < 10 else d.year + 1
)

# 3. Calculate annual revenue from quarters
annual_totals = (
    df_quarterly.groupby('Fiscal Year', as_index=False)['Net Income (USD)'].sum()
    .rename(columns={'Net Income (USD)': 'Fiscal Year Revenue'})
)

# 4. Merge the totals back into df_quarterly
df_quarterly = df_quarterly.merge(annual_totals, on='Fiscal Year', how='left')

# Optional: sort by date
net_income = df_quarterly.sort_values(['Fiscal Year', 'Period End']).reset_index(drop=True)


net_income[30:]

#Q1 2012 is missing in the data

,Period Start,Period End,Net Income (USD),Tag Used,Period Type,FP,Form,Period Length (days),Fiscal Year,Fiscal Year Revenue
30,2016-03-27,2016-06-25,7796000000,NetIncomeLoss,Quarterly,Q3,10-Q,90,2016,45687000000
31,2016-06-26,2016-09-24,9014000000,NetIncomeLoss,Quarterly,FY,10-K,90,2016,45687000000
32,2017-01-01,2017-04-01,11029000000,NetIncomeLoss,Quarterly,Q2,10-Q,90,2017,30460000000
33,2017-04-02,2017-07-01,8717000000,NetIncomeLoss,Quarterly,Q3,10-Q,90,2017,30460000000
34,2017-07-02,2017-09-30,10714000000,NetIncomeLoss,Quarterly,FY,10-K,90,2017,30460000000
35,2017-10-01,2017-12-30,20065000000,NetIncomeLoss,Quarterly,FY,10-K,90,2018,59531000000
36,2017-12-31,2018-03-31,13822000000,NetIncomeLoss,Quarterly,FY,10-K,90,2018,59531000000
37,2018-04-01,2018-06-30,11519000000,NetIncomeLoss,Quarterly,FY,10-K,90,2018,59531000000
38,2018-07-01,2018-09-29,14125000000,NetIncomeLoss,Quarterly,FY,10-K,90,2018,59531000000
39,2018-09-30,2018-12-29,19965000000,NetIncomeLoss,Quarterly,FY,10-K,90,2019,55256000000


In [31]:
#Check for missing quarters

counts = net_income.groupby('Fiscal Year').size()
print(counts)

Fiscal Year
2008    1
2009    4
2010    4
2011    4
2012    3
2013    4
2014    4
2015    4
2016    4
2017    3
2018    4
2019    4
2020    4
2021    3
2022    3
2023    2
2024    3
2025    3
dtype: int64
